# Omni ChromHMM SAGAconf Analysis

## Compute

In [1]:
import os
import sys
import yaml
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm
import importlib
from IPython.display import display
import glob
from collections import defaultdict
from concurrent.futures import ProcessPoolExecutor, as_completed
from itertools import combinations

In [2]:
# Load configuration and find project root
if "PROJECT_ROOT" not in globals():
    PROJECT_ROOT = None
    for _p in [os.getcwd()] + sys.path:
        _curr = os.path.abspath(_p) if _p else ""
        while _curr and _curr != os.path.dirname(_curr):
            if os.path.exists(os.path.join(_curr, "scripts/analysis/analyze.py")):
                PROJECT_ROOT = _curr
                break
            _curr = os.path.dirname(_curr)
        if PROJECT_ROOT: break
    if not PROJECT_ROOT:
        PROJECT_ROOT = os.getcwd()

config_path = os.path.join(PROJECT_ROOT, "config_sagaconf.yaml")
with open(config_path, "r") as f:
    config = yaml.safe_load(f)

scripts_dir = os.path.abspath(os.path.join(PROJECT_ROOT, "scripts", "analysis"))
scripts_rules_dir = os.path.abspath(os.path.join(PROJECT_ROOT, "scripts", "rules"))
scripts_root_dir = os.path.abspath(os.path.join(PROJECT_ROOT, "scripts"))
workdir = os.path.expanduser(config.get("workdir", "."))

In [3]:
for _p in [scripts_root_dir, scripts_rules_dir, scripts_dir]:
    if _p not in sys.path:
        sys.path.insert(0, _p)

import match
import utils
from utils import (
    CHROMHMM, CHROMHMM_DEFAULT,
    AMI, AMI_DISPLAY, COSINE, COSINE_DISPLAY,
    HOMER, HOMER_DISPLAY, JACCARD, JACCARD_DISPLAY,
    JOINT_CHROMHMM, JOINT_KMEANS_HOMER, JOINT_KMEANS_MACS2, JOINT_KMEANS_OMNI,
    KAPPA, KAPPA_DISPLAY, KMEANS_HOMER, KMEANS_MACS2, KMEANS_OMNI,
    MACS2, MACS2_DISPLAY, OMNI, OMNI_DISPLAY,
    QUIESCENT
)
import analyze
import analyze_peaks
import compare
import compare_methods
import summary_plots
import interpretation

# Re-import the analysis modules
for _m in (analyze, analyze_peaks, match, compare, compare_methods, summary_plots, utils,
           interpretation):
    importlib.reload(_m)

# The project's plot style (fonts, PDF output) from utils, set once for every
# plot below; after the reload, so a re-run picks up edits to utils.py.
utils.set_style()

In [4]:
method_palette = {utils.display_name(m): utils.method_color(m) for m in [
    CHROMHMM_DEFAULT, KMEANS_HOMER, KMEANS_MACS2, KMEANS_OMNI,
    JOINT_CHROMHMM,
    JOINT_KMEANS_HOMER, JOINT_KMEANS_MACS2, JOINT_KMEANS_OMNI,
]}

os.chdir(workdir)
os.makedirs("out/summary_plots", exist_ok=True)
print(f"Project root: {PROJECT_ROOT}")
print(f"Workdir     : {workdir}")

P = config["params"]
DATASETS = config["datasets"]
# The config's outliers, if it names any, are left out of the analysis.
OUTLIERS = config.get("outliers") or []
DATASETS = {k: v for k, v in DATASETS.items() if k not in OUTLIERS}
NSTATES = P["n_states"]
MATCH_METHOD = "matched"

# Grouped by caller: a plot of many methods lays its x axis out as ChromHMM,
# HOMER, MACS2, OmniPeak, the models of one caller standing together under one
# group label rather than repeating the caller on every tick
# (utils.group_methods(), utils.group_xticks()).
METHOD_LABELS = utils.group_methods([
    (CHROMHMM_DEFAULT, utils.display_name(CHROMHMM_DEFAULT)),
    (KMEANS_HOMER, utils.display_name(KMEANS_HOMER)),
    (KMEANS_MACS2, utils.display_name(KMEANS_MACS2)),
    (KMEANS_OMNI, utils.display_name(KMEANS_OMNI)),
    (JOINT_CHROMHMM, utils.display_name(JOINT_CHROMHMM)),
    (JOINT_KMEANS_HOMER, utils.display_name(JOINT_KMEANS_HOMER)),
    (JOINT_KMEANS_MACS2, utils.display_name(JOINT_KMEANS_MACS2)),
    (JOINT_KMEANS_OMNI, utils.display_name(JOINT_KMEANS_OMNI)),
], key=lambda item: item[0])
METHOD_ORDER = [m[1] for m in METHOD_LABELS]

def get_method_path(ds, method, rep=None):
    folder = ds
    if rep:
        folder = f"{ds}/{rep}"

    if method == CHROMHMM_DEFAULT:
        cell = DATASETS[ds]["cell"]
        return f"{folder}/{CHROMHMM_DEFAULT}_result/{cell}_{NSTATES}_dense_matched.bed"
    if method == JOINT_CHROMHMM:
        return f"{ds}/{JOINT_CHROMHMM}/{rep or 'rep1'}_{NSTATES}_dense.bed"
    if method.startswith("kmeans_"):
        caller = method.replace("kmeans_", "")
        return f"{folder}/{caller}/{caller}_kmeans_states_matched.bed"
    if method.startswith("joint_kmeans_"):
        caller = method.replace("joint_kmeans_", "")
        return f"{ds}/joint_kmeans/{caller}/{rep or 'rep1'}_kmeans_joint_states.bed"
    return None

CHROMHMM_BIN = P.get("chromhmm_bin", utils.CHROMHMM_BIN)
PEAKS_BIN = P.get("peaks_bin", utils.PEAKS_BIN)


def dataset_outdir(ds):
    """Directory holding the aggregated results of one dataset."""
    return f"out/{ds}/{MATCH_METHOD}"


def method_outdir(ds, method_key, rep):
    """Analysis directory of a single segmentation."""
    outdir = f"{dataset_outdir(ds)}/{method_key}"
    return outdir if rep == "rep1" else f"{outdir}_{rep}"


def dataset_reps(ds):
    """Replicates of a dataset: rep1 and rep2, or rep1 alone."""
    return ["rep1", "rep2"] if DATASETS[ds].get("replicates", True) else ["rep1"]


def dataset_segmentations(ds):
    """(method_key, method_name, rep, path) of every segmentation of ds on disk."""
    result = []
    for method_key, method_name in METHOD_LABELS:
        for rep in dataset_reps(ds):
            path = get_method_path(ds, method_key, rep)
            if path and os.path.exists(path):
                result.append((method_key, method_name, rep, path))
    return result


# Shared analysis helpers

def nonempty(df):
    """Cache guard: an empty table means the inputs were not on disk yet."""
    return not df.empty


# Bump after changing match.agreement_metrics() to invalidate every cached
# agreement table; agreement_rows() stamps the tables it builds with it.
AGREEMENT_CACHE_VERSION = 2


def has_metrics(df):
    """Cache guard of an agreement table: nonempty, holding every metric
    match.agreement_metrics() reports, and built by its current version - a
    cache from before the composition cosine left the biggest state out is
    not. The AMI compares the two segmentations as partitions and ignores the
    state names, so it needs neither the matching to the joint model nor the
    emission rematching of a cross-dataset pair."""
    return (nonempty(df) and AMI_DISPLAY in df.columns
            and df.attrs.get("version") == AGREEMENT_CACHE_VERSION)


def load_per_dataset_tsv(filename):
    """Concatenated out/<ds>/matched/<filename> over every dataset.

    Adds the Dataset column and maps each segmentation label to its display
    Method, keeping only the methods of METHOD_ORDER. Empty until run_analysis()
    below has aggregated at least one dataset.
    """
    frames = []
    for ds in DATASETS:
        path = f"{dataset_outdir(ds)}/{filename}"
        if os.path.exists(path):
            df = pd.read_csv(path, sep="\t")
            df["Dataset"] = ds
            frames.append(df)
    if not frames:
        return pd.DataFrame()
    df = pd.concat(frames, ignore_index=True)
    # compare.run_compare() labels a segmentation "{method_key}_{rep}", so the
    # replicate is stripped before the method is resolved: display_name() has
    # no entry for a joint key carrying a rep suffix and would leave it
    # unmapped, dropping every joint row in the filter below.
    labels = df["segmentation"].astype(str)
    df["Replicate"] = labels.str.extract(r"_(rep\d)$", expand=False).fillna("rep1")
    df["Method"] = (labels.str.replace(r"_rep\d$", "", regex=True)
                    .map(utils.normalize_method).map(utils.display_name))
    return df[df["Method"].isin(METHOD_ORDER)]


def load_bin_emissions(seg_path):
    """(states, marks, matrix) of a segmentation's binarized emissions, or None."""
    path = seg_path.replace(".bed", ".bin_emissions.npz")
    if not os.path.exists(path):
        return None
    data = np.load(path, allow_pickle=False)
    return ([str(s) for s in data["states"]],
            [str(m) for m in data["marks"]],
            data["mat"].astype(float))


def emission_state_mapping(from_path, to_path):
    """One-to-one state mapping from_path -> to_path, by emission cosine.

    process_sagaconf.sh matches every model to the joint model of its own
    dataset, and each of those numbers its states arbitrarily, so state "13"
    of MCF7 has nothing to do with state "13" of K562: comparing two datasets
    by state name scores near zero whatever the method. The Hungarian match on
    the binarized emission vectors puts the two state spaces back in
    correspondence first. Only the marks both models were trained on take
    part, so a dataset missing an assay does not turn every cosine into a NaN,
    and match.background_augmented(), which emission_cosine_mapping() applies
    itself, keeps the match from scattering the states that carry no marks.

    Needed for a cross-dataset pair and for nothing else. Both replicates of a
    dataset, and its individual and joint models, were matched to that
    dataset's own joint model by match.py, which pairs states by genomic
    overlap - a stronger correspondence than an emission cosine, and one that
    rematching would replace rather than improve.

    What it cannot repair is a pair of models that split the background into a
    different number of states - HeLa-S3 ChromHMM has three quiescent states to
    CD14 monocyte's two - since then no one-to-one mapping of the two state
    spaces exists to be found.
    """
    src, dst = load_bin_emissions(from_path), load_bin_emissions(to_path)
    if src is None or dst is None:
        return None
    states_src, marks_src, mat_src = src
    states_dst, marks_dst, mat_dst = dst
    shared = [m for m in marks_src if m in set(marks_dst)]
    if not shared:
        return None
    _, mapping = match.emission_cosine_mapping(
        states_src, mat_src[:, [marks_src.index(m) for m in shared]],
        states_dst, mat_dst[:, [marks_dst.index(m) for m in shared]])
    return mapping


def compute_metrics(s1_path, s2_path, rematch=False):
    """Agreement of two segmentations, None when they cannot be compared.

    The metrics themselves come from match.agreement_metrics(); a pair that
    shares no state or never overlaps is reported as None and left out of the
    plots, rather than entering them as a row of zeros.

    rematch=True relabels s2 into the state space of s1 first, for a pair that
    does not already share one - see emission_state_mapping(). A pair whose
    emissions are not on disk is reported as None rather than compared by a
    state numbering that means nothing across the two sides.
    """
    if not os.path.exists(s1_path) or not os.path.exists(s2_path):
        return None
    s1, s2 = match.load_bed(s1_path), match.load_bed(s2_path)
    if rematch:
        mapping = emission_state_mapping(s2_path, s1_path)
        if mapping is None:
            return None
        s2 = [(chrom, start, end, mapping.get(state, state), color)
              for chrom, start, end, state, color in s2]
    l1, l2 = match.state_lengths(s1), match.state_lengths(s2)
    # pair_overlap() puts its *work* state first, so s2 goes in as the reference
    # and the keys come out (s1_state, s2_state), matching the l1 / l2 order.
    overlap = match.pair_overlap(s2, s1)
    states = set(l1) | set(l2)
    if not states or sum(overlap.get((a, b), 0) for a in states for b in states) == 0:
        return None
    return match.agreement_metrics(overlap, l1, l2)


def agreement_rows(pairs, desc, rematch=False):
    """DataFrame of compute_metrics() over (method_name, dataset, path1, path2)."""
    rows = []
    for method_name, ds, path1, path2 in tqdm(pairs, desc=desc):
        metrics = compute_metrics(path1, path2, rematch=rematch)
        if metrics:
            rows.append({"Method": method_name, "Dataset": ds,
                         **{utils.metric_display(m): v for m, v in metrics.items()}})
    df = pd.DataFrame(rows)
    df.attrs["version"] = AGREEMENT_CACHE_VERSION
    return df


Project root: /Users/Oleg.Shpynov/work/omni-chromhmm
Workdir     : /Users/Oleg.Shpynov/data/2026_segmentations/sagaconf


## 5. General processing (emissions and enrichment)

In [5]:
# Configuration for analysis
COORDS_DIR = config["tools"].get("coords_dir")
annotations = sorted(glob.glob(os.path.join(workdir, COORDS_DIR, "*.bed.gz"))) if COORDS_DIR else []

# Bump after changing analyze.py / compare.py to invalidate every cached aggregation.
ANALYSIS_CACHE_VERSION = 1
# Produced by compare.run_compare and consumed by the plots below; a missing one
# invalidates the stamp, so deleting a result is enough to force a rerun.
COMPARE_OUTPUTS = ["segment_stats.tsv", "entropy_summary.tsv"]


def compare_signature(ds):
    """Identity of every input compare.run_compare() reads for this dataset:
    the segmentations, their emission matrices and their per-state reports."""
    items = []
    for method_key, _, rep, path in dataset_segmentations(ds):
        items.append({
            "label": f"{method_key}_{rep}",
            "bin": utils.seg_bin_size(path),
            "seg": utils.file_stamp(path),
            "bin_emissions": utils.file_stamp(path.replace(".bed", ".bin_emissions.npz")),
            "bw_emissions": utils.file_stamp(path.replace(".bed", ".bw_emissions.npz")),
            "report": utils.file_stamp(f"{method_outdir(ds, method_key, rep)}/report.tsv"),
        })
    return {"version": ANALYSIS_CACHE_VERSION, "items": items}


def run_analysis(force=False):
    """Analyze every segmentation, then aggregate the results per dataset.

    Both steps are cached, so re-running the cell only computes what is missing:
    a segmentation is skipped once its report.tsv is there, and the (expensive)
    aggregation of a dataset is skipped while its inputs are unchanged since the
    stamp of the cached run, out/<ds>/matched/compare_stamp.json.
    force=True recomputes everything.
    """
    futures = {}
    reanalyzed = set()   # datasets with a fresh analysis: aggregation must rerun
    # We use a subset of markers if necessary, but here we assume everything is set up.
    with ProcessPoolExecutor(max_workers=max(1, os.cpu_count() // 2)) as executor:
        for ds in DATASETS:
            for method_key, method_name, rep, path in dataset_segmentations(ds):
                outdir = method_outdir(ds, method_key, rep)
                if not force and os.path.exists(f"{outdir}/report.tsv"):
                    continue
                print(f"Analyzing {method_name} ({rep}) for {ds}...")

                # Determine inputs (binarized data)
                inputs = None
                if CHROMHMM in method_key:
                    inputs = [f"{ds}/{rep}/chromhmm_default/*.txt"]
                elif "kmeans" in method_key:
                    caller = method_key.replace("joint_", "").replace("kmeans_", "")
                    inputs = [f"{ds}/{rep}/{caller}/chromhmm_peaks/chr*.txt.gz"]

                reanalyzed.add(ds)
                fut = executor.submit(
                    analyze.run_analyze,
                    seg=path, bin_size=utils.seg_bin_size(path),
                    outdir=outdir,
                    inputs=inputs,
                    annotations=annotations,
                    bw_emissions=path.replace(".bed", ".bw_emissions.npz"),
                    emissions_only=False,
                )
                futures[fut] = f"{method_name} ({rep}) for {ds}"

    for fut in as_completed(futures):
        try:
            fut.result()
        except Exception as e:
            print(f"  ERROR: {futures[fut]}: {e}")

    # 2. Aggregation step: run compare.run_compare to produce segment_stats.tsv and entropy_summary.tsv
    for ds in DATASETS:
        segmentations = dataset_segmentations(ds)
        if not segmentations:
            continue

        comp_outdir = dataset_outdir(ds)
        stamp_path = f"{comp_outdir}/compare_stamp.json"
        # Stamped before the run, so a segmentation that changes while compare is
        # running is not recorded as already aggregated.
        signature = compare_signature(ds)
        outputs = [f"{comp_outdir}/{name}" for name in COMPARE_OUTPUTS]
        if not force and ds not in reanalyzed and utils.stamp_current(stamp_path, signature, outputs):
            print(f"Aggregation for {ds} is up to date, skipped.")
            continue

        segs = [path for _, _, _, path in segmentations]
        bins = [utils.seg_bin_size(path) for _, _, _, path in segmentations]
        # Important: labels must follow the _rep1 / _rep2 convention for should_compare to work
        labels = [f"{method_key}_{rep}" for method_key, _, rep, _ in segmentations]

        print(f"Aggregating results for {ds}...")
        compare.run_compare(seg=segs, bins=bins, labels=labels, all_pairs=False,
                            outdir=comp_outdir, analysis_dir=comp_outdir)
        utils.save_stamp(stamp_path, signature)


# Run analysis if needed
run_analysis()


Aggregation for mcf7 is up to date, skipped.
Aggregation for gm12878 is up to date, skipped.
Aggregation for k562 is up to date, skipped.
Aggregation for cd14_monocyte is up to date, skipped.
Aggregation for hela_s3 is up to date, skipped.


## 0. Peak calling stats per methods

In [6]:
def compute_peak_stats():
    frames = []
    for ds in DATASETS:
        p = f"{ds}/peaks/peak_stats.tsv"
        if os.path.exists(p):
            df = pd.read_csv(p, sep='\t')
            # Filter marks based on DATASETS[ds]['marks'] if available
            if 'marks' in DATASETS[ds]:
                df = df[df['mark'].isin(DATASETS[ds]['marks'])]
            df['Dataset'] = ds
            frames.append(df)
    if not frames:
        return pd.DataFrame()
    df_all = pd.concat(frames, ignore_index=True)
    df_all['method'] = df_all['method'].replace('Default', utils.display_name(CHROMHMM_DEFAULT))
    df_all['method'] = df_all['method'].replace(HOMER_DISPLAY, utils.display_name(KMEANS_HOMER))
    df_all['method'] = df_all['method'].replace(MACS2_DISPLAY, utils.display_name(KMEANS_MACS2))
    df_all['method'] = df_all['method'].replace(OMNI_DISPLAY, utils.display_name(KMEANS_OMNI))

    return df_all


df_peaks = utils.cached_pickle("out/df_peaks.pkl", compute_peak_stats,
                               label="peak statistics", valid=nonempty)

if not df_peaks.empty:

    order = utils.group_methods([
        utils.display_name(CHROMHMM_DEFAULT),
        utils.display_name(KMEANS_HOMER),
        utils.display_name(KMEANS_MACS2),
        utils.display_name(KMEANS_OMNI)
    ])
    marks_present = df_peaks['mark'].unique()
    # Canonical order from summary_plots.py
    MARK_ORDER = ["H3K4me3", "H3K27ac", "H3K4me1", "H3K36me3", "H3K9me3", "H3K27me3"]
    marks = [m for m in MARK_ORDER if m in marks_present]
    marks += sorted([m for m in marks_present if m not in MARK_ORDER])

    # Plot 1: Number of peaks
    plt.figure(figsize=(max(8, len(marks) * 0.8), 5))
    sns.barplot(data=df_peaks, x='mark', y='n_peaks', order=marks,
                hue='method', hue_order=order, palette=method_palette,
                **utils.BAR_STYLE)
    plt.title("Number of peaks per mark", **utils.TITLE_STYLE)
    plt.ylabel("Count", fontsize=utils.AXIS_FONTSIZE)
    plt.grid(axis='y', alpha=0.3)
    utils.strip_points(plt.gca(), data=df_peaks, x='mark', y='n_peaks', order=marks,
                       hue='method', hue_order=order, size=2)
    plt.xticks(rotation=45, ha="right", fontsize=utils.TICK_FONTSIZE)
    plt.yticks(fontsize=utils.TICK_FONTSIZE)
    plt.legend(title="Method", fontsize=utils.LEGEND_FONTSIZE, title_fontsize=utils.LEGEND_TITLE_FONTSIZE, bbox_to_anchor=(1.01, 1), loc="upper left")
    utils.save_fig(plt.gcf(), "out/summary_plots/n_peaks.pdf")

    # Plot 2: Average peak length
    plt.figure(figsize=(max(8, len(marks) * 0.8), 5))
    sns.barplot(data=df_peaks, x='mark', y='mean_length', order=marks,
                hue='method', hue_order=order, palette=method_palette,
                **utils.BAR_STYLE)
    plt.title("Average peak length per mark", **utils.TITLE_STYLE)
    plt.ylabel("Length (bp)", fontsize=utils.AXIS_FONTSIZE)
    plt.grid(axis='y', alpha=0.3)
    utils.strip_points(plt.gca(), data=df_peaks, x='mark', y='mean_length', order=marks,
                       hue='method', hue_order=order, size=2)
    plt.xticks(rotation=45, ha="right", fontsize=utils.TICK_FONTSIZE)
    plt.yticks(fontsize=utils.TICK_FONTSIZE)
    plt.legend(title="Method", fontsize=utils.LEGEND_FONTSIZE, title_fontsize=utils.LEGEND_TITLE_FONTSIZE, bbox_to_anchor=(1.01, 1), loc="upper left")
    utils.save_fig(plt.gcf(), "out/summary_plots/peak_length.pdf")

    # Plot 3: Binarization mark coverage
    datasets_11 = [d for d in DATASETS if len(DATASETS[d].get("marks", {b["mark"] for b in DATASETS[d].get("bams", {}).values() if isinstance(b, dict) and "mark" in b})) == 11]
    summary_plots._plot_mark_coverage(
        datasets_11, workdir,
        "out/summary_plots/binarization_mark_coverage_absolute.pdf",
        cells=[DATASETS[d]["cell"] for d in datasets_11],
        relative=False
    )
    summary_plots._plot_mark_coverage(
        datasets_11, workdir,
        "out/summary_plots/binarization_mark_coverage_relative.pdf",
        cells=[DATASETS[d]["cell"] for d in datasets_11],
        relative=True
    )
    summary_plots._plot_mark_combinations(
        datasets_11, workdir,
        "out/summary_plots/binarization_mark_combinations.pdf",
        cells=[DATASETS[d]["cell"] for d in datasets_11]
    )

Computing peak statistics...
Saved peak statistics to out/df_peaks.pkl


  saved out/summary_plots/n_peaks.pdf


  saved out/summary_plots/peak_length.pdf


  saved out/summary_plots/binarization_mark_coverage_absolute.pdf


  saved out/summary_plots/binarization_mark_coverage_relative.pdf


  saved out/summary_plots/binarization_mark_combinations.pdf


## 1. Number of segments stats per methods

In [7]:
df_segs = utils.cached_pickle(
    "out/df_segs.pkl", lambda: load_per_dataset_tsv("segment_stats.tsv"),
    label="segment statistics", valid=nonempty)

if not df_segs.empty:

    plt.figure(figsize=(12, 5))
    sns.barplot(data=df_segs, x='Method', y='n_segments', order=METHOD_ORDER,
                palette=method_palette, hue='Method', hue_order=METHOD_ORDER, dodge=False,
                legend=False, **utils.BAR_STYLE)
    utils.hatch_joint(plt.gca(), METHOD_ORDER)
    utils.strip_points(plt.gca(), data=df_segs, x='Method', y='n_segments', order=METHOD_ORDER, size=2)
    plt.title("Number of segments per method", **utils.TITLE_STYLE)
    plt.ylabel("Number of segments", fontsize=utils.AXIS_FONTSIZE)
    utils.group_xticks(plt.gca(), METHOD_ORDER, bands=False, fontsize=utils.TICK_FONTSIZE)
    plt.yticks(fontsize=utils.TICK_FONTSIZE)
    plt.grid(axis='y', alpha=0.3)
    yrange = plt.gca().get_ylim()[1] - plt.gca().get_ylim()[0]
    for i, method in enumerate(METHOD_ORDER):
        vals = df_segs[df_segs["Method"] == method]["n_segments"]
        if vals.empty: continue
        m, s = vals.mean(), vals.sem()
        if pd.isna(m): continue
        top = max(m + (s if not pd.isna(s) else 0), vals.max())
        plt.text(i, top + yrange * 0.01, f"{int(m)}", ha="center", va="bottom", fontsize=utils.LABEL_FONTSIZE)
    utils.save_fig(plt.gcf(), "out/summary_plots/n_segments.pdf")

Computing segment statistics...
Saved segment statistics to out/df_segs.pkl
  saved out/summary_plots/n_segments.pdf


## 2. Transition matrix entropy per method

In [8]:
df_entropy = utils.cached_pickle(
    "out/df_entropy.pkl", lambda: load_per_dataset_tsv("entropy_summary.tsv"),
    label="transition matrix entropy", valid=nonempty)

if not df_entropy.empty:

    plt.figure(figsize=(12, 5))
    sns.barplot(data=df_entropy, x='Method', y='total_entropy', order=METHOD_ORDER,
                palette=method_palette, hue='Method', hue_order=METHOD_ORDER, dodge=False,
                legend=False, **utils.BAR_STYLE)
    utils.hatch_joint(plt.gca(), METHOD_ORDER)
    utils.strip_points(plt.gca(), data=df_entropy, x='Method', y='total_entropy', order=METHOD_ORDER, size=2)
    plt.title("Transition matrix entropy per method", **utils.TITLE_STYLE)
    plt.ylabel("Entropy (bits)", fontsize=utils.AXIS_FONTSIZE)
    utils.group_xticks(plt.gca(), METHOD_ORDER, bands=False, fontsize=utils.TICK_FONTSIZE)
    plt.yticks(fontsize=utils.TICK_FONTSIZE)
    plt.grid(axis='y', alpha=0.3)
    yrange = plt.gca().get_ylim()[1] - plt.gca().get_ylim()[0]
    for i, method in enumerate(METHOD_ORDER):
        vals = df_entropy[df_entropy["Method"] == method]["total_entropy"]
        if vals.empty: continue
        m, s = vals.mean(), vals.sem()
        if pd.isna(m): continue
        top = max(m + (s if not pd.isna(s) else 0), vals.max())
        plt.text(i, top + yrange * 0.01, f"{m:.2f}", ha="center", va="bottom", fontsize=utils.LABEL_FONTSIZE)
    utils.save_fig(plt.gcf(), "out/summary_plots/entropy.pdf")

Computing transition matrix entropy...
Saved transition matrix entropy to out/df_entropy.pkl


  saved out/summary_plots/entropy.pdf


## 3. State composition and lengths

In [9]:
# Collect valid tasks for state-based analysis
valid_tasks = [(f"{ds}_{rep}", method_name, path)
               for ds in DATASETS
               for _, method_name, rep, path in dataset_segmentations(ds)]

def state_comp_worker(task):
    folder, method_name, path = task
    try:
        segs = match.load_bed(path)
        lengths_by_state = defaultdict(list)
        for row in segs:
            name = row[3]
            if not name or name == ".": continue
            if "Unknown" in name:
                raise ValueError(f"Unknown state detected in {path}: {name}")
            lengths_by_state[name].append(row[2] - row[1])

        tot_bp = sum(sum(ls) for ls in lengths_by_state.values())

        res = []
        for st, ls in lengths_by_state.items():
            res.append({
                "Dataset": folder,
                "Method": method_name,
                "State": st,
                "Fraction": sum(ls) / tot_bp if tot_bp > 0 else 0,
                "MeanLength": np.mean(ls),
                "MedianLength": np.median(ls)
            })
        return res
    except Exception as e:
        return []

def compute_state_composition():
    """(composition table, {state: hex color}) over every segmentation."""
    per_task = [state_comp_worker(t) for t in tqdm(valid_tasks)]
    df = pd.DataFrame([row for rows in per_task for row in rows])

    # State colors, from the itemRgb column of the segmentation BEDs
    colors_hex = {}
    for _, _, path in tqdm(valid_tasks, desc="Extracting state colors"):
        if not os.path.exists(path): continue
        for row in match.load_bed(path):
            name = row[3]
            color = row[8] if len(row) > 8 else None
            if name not in colors_hex and color and color != "0,0,0":
                colors_hex[name] = analyze.rgb_str_to_hex(color)
    return df, colors_hex


df_comp, state_colors_hex = utils.cached_pickle(
    "out/df_comp.pkl", compute_state_composition,
    label="state composition and lengths")

state_colors_hex = summary_plots.state_palette(df_comp["State"].unique(), state_colors_hex)
states_order = summary_plots.sort_states(df_comp["State"].unique())
# Plot 3a. Average state composition per method (broken axis)
df_comp_filled = df_comp.pivot_table(index=["Method", "Dataset"], columns="State", values="Fraction", fill_value=0).stack().reset_index(name="Fraction")

BREAK_LOW = 0.20
BREAK_HIGH = 0.40
figw = max(12, len(states_order) * len(METHOD_ORDER) * 0.22)

fig, (ax_top, ax_bot) = plt.subplots(
    2, 1, sharex=True,
    figsize=(figw, 6),
    gridspec_kw={"height_ratios": [1, 4], "hspace": 0.06},
)

for ax in (ax_top, ax_bot):
    sns.barplot(
        data=df_comp_filled, x="State", y="Fraction", hue="Method",
        order=states_order, hue_order=METHOD_ORDER, palette=method_palette,
        ax=ax, legend=(ax is ax_top), **utils.BAR_STYLE)
    utils.hatch_joint(ax, METHOD_ORDER)
    utils.strip_points(ax, data=df_comp_filled, x="State", y="Fraction", hue="Method",
                       order=states_order, hue_order=METHOD_ORDER,
                       size=1.5, alpha=0.4, jitter=0.2)

ax_top.set_ylim(BREAK_HIGH, 1.02)
ax_bot.set_ylim(0, BREAK_LOW)
ax_top.spines["bottom"].set_visible(False)
ax_bot.spines["top"].set_visible(False)
ax_top.tick_params(axis="x", bottom=False)

for ax in (ax_top, ax_bot):
    utils.widen_bars(ax)
utils.break_marks(ax_top, ax_bot)

for ax in (ax_top, ax_bot):
    ax.grid(axis="y", alpha=0.3, linewidth=0.5)
    ax.tick_params(axis="y", labelsize=utils.TICK_FONTSIZE)

ax_bot.tick_params(axis="x", labelsize=utils.TICK_FONTSIZE, rotation=45)
ax_bot.set_xlabel("Chromatin state", fontsize=utils.AXIS_FONTSIZE)
ax_top.set_xlabel("")
ax_top.set_title("Average state composition per method", **utils.TITLE_STYLE)
ax_top.legend(title="Method", fontsize=utils.LEGEND_FONTSIZE, title_fontsize=utils.LEGEND_TITLE_FONTSIZE,
              bbox_to_anchor=(1.01, 1), loc="upper left", borderaxespad=0)
if ax_bot.get_legend(): ax_bot.get_legend().remove()
ax_bot.set_ylabel("Fraction of genome", fontsize=utils.AXIS_FONTSIZE)
ax_top.set_ylabel("")

utils.save_fig(fig, "out/summary_plots/avg_composition.pdf", tight=False)

# Plot 3b. Average state mean length per method
plt.figure(figsize=(figw, 6))
ax = plt.gca()
sns.barplot(
    data=df_comp, x="State", y="MeanLength", hue="Method",
    order=states_order, hue_order=METHOD_ORDER, palette=method_palette,
    ax=ax, **utils.BAR_STYLE)
utils.hatch_joint(ax, METHOD_ORDER)
utils.strip_points(ax, data=df_comp, x="State", y="MeanLength", hue="Method",
                   order=states_order, hue_order=METHOD_ORDER,
                   size=1.5, alpha=0.4, jitter=0.2)
if not df_comp.empty and (df_comp["MeanLength"] > 0).any():
    ax.set_yscale("log")
ax.set_title("Average state mean length per method", **utils.TITLE_STYLE)
ax.set_ylabel("Mean length (bp, log scale)", fontsize=utils.AXIS_FONTSIZE)
ax.set_xlabel("Chromatin state", fontsize=utils.AXIS_FONTSIZE)
ax.grid(axis="y", alpha=0.3, linewidth=0.5)
ax.tick_params(axis="x", labelsize=utils.TICK_FONTSIZE, rotation=45)
ax.tick_params(axis="y", labelsize=utils.TICK_FONTSIZE)
ax.legend(title="Method", fontsize=utils.LEGEND_FONTSIZE, title_fontsize=utils.LEGEND_TITLE_FONTSIZE,
          bbox_to_anchor=(1.01, 1), loc="upper left", borderaxespad=0)
utils.save_fig(plt.gcf(), "out/summary_plots/avg_mean_length.pdf")

# Plot 3c. Average state median length per method
plt.figure(figsize=(figw, 6))
ax = plt.gca()
sns.barplot(
    data=df_comp, x="State", y="MedianLength", hue="Method",
    order=states_order, hue_order=METHOD_ORDER, palette=method_palette,
    ax=ax, **utils.BAR_STYLE)
utils.hatch_joint(ax, METHOD_ORDER)
utils.strip_points(ax, data=df_comp, x="State", y="MedianLength", hue="Method",
                   order=states_order, hue_order=METHOD_ORDER,
                   size=1.5, alpha=0.4, jitter=0.2)
if not df_comp.empty and (df_comp["MedianLength"] > 0).any():
    ax.set_yscale("log")
ax.set_title("Average state median length per method", **utils.TITLE_STYLE)
ax.set_ylabel("Median length (bp, log scale)", fontsize=utils.AXIS_FONTSIZE)
ax.set_xlabel("Chromatin state", fontsize=utils.AXIS_FONTSIZE)
ax.grid(axis="y", alpha=0.3, linewidth=0.5)
ax.tick_params(axis="x", labelsize=utils.TICK_FONTSIZE, rotation=45)
ax.tick_params(axis="y", labelsize=utils.TICK_FONTSIZE)
ax.legend(title="Method", fontsize=utils.LEGEND_FONTSIZE, title_fontsize=utils.LEGEND_TITLE_FONTSIZE,
          bbox_to_anchor=(1.01, 1), loc="upper left", borderaxespad=0)
utils.save_fig(plt.gcf(), "out/summary_plots/avg_median_length.pdf")

# Plot 3d. Summary average state composition per method (stacked)
pivot_avg = df_comp.pivot_table(index=["Method", "Dataset"], columns="State", values="Fraction", fill_value=0).groupby("Method").mean()
pivot_avg = pivot_avg.reindex(METHOD_ORDER)
avail_states = [s for s in states_order if s in pivot_avg.columns]
pivot_avg = pivot_avg[avail_states]
colors = [state_colors_hex.get(s, "#888888") for s in avail_states]

sums = pivot_avg.sum(axis=1)
if not np.allclose(sums[sums > 0], 1.0, atol=1e-5):
    raise ValueError("Average composition not normalized to 1.0")

plt.figure(figsize=(8, 5))
ax = pivot_avg.plot(kind="bar", stacked=True, color=colors, width=0.6, ax=plt.gca(), linewidth=0)
pivot_avg.sum(axis=1).plot(kind="bar", ax=ax, width=0.6, facecolor="none", edgecolor=utils.BAR_STYLE["edgecolor"], linewidth=utils.BAR_STYLE["linewidth"], legend=False)
ax.set_title("Average state composition per method", **utils.TITLE_STYLE)
ax.legend(bbox_to_anchor=(1.01, 1), loc="upper left", fontsize=utils.LEGEND_FONTSIZE, title_fontsize=utils.LEGEND_TITLE_FONTSIZE, title="State")
ax.set_xlabel("Method", fontsize=utils.AXIS_FONTSIZE)
ax.set_ylabel("Average Fraction of Genome", fontsize=utils.AXIS_FONTSIZE)
utils.group_xticks(ax, list(pivot_avg.index), bands=False, fontsize=utils.TICK_FONTSIZE)
ax.tick_params(axis="y", labelsize=utils.TICK_FONTSIZE)
ax.grid(axis="y", alpha=0.3, linewidth=0.5)
utils.widen_bars(ax)
utils.save_fig(plt.gcf(), "out/summary_plots/avg_composition_stacked.pdf")

# Plot 3e. State composition per dataset for each method
for method in METHOD_ORDER:
    method_df = df_comp[df_comp["Method"] == method]
    if method_df.empty: continue
    pivot_df = method_df.pivot(index="Dataset", columns="State", values="Fraction").fillna(0)
    avail_states = [s for s in states_order if s in pivot_df.columns]
    pivot_df = pivot_df[avail_states]
    colors = [state_colors_hex.get(s, "#888888") for s in avail_states]

    if not np.allclose(pivot_df.sum(axis=1), 1.0, atol=1e-5):
        raise ValueError(f"Composition for method {method} not normalized to 1.0")

    ax = pivot_df.plot(kind="bar", stacked=True, figsize=(max(10, len(pivot_df)*0.5), 6), color=colors, width=0.8, linewidth=0)
    pivot_df.sum(axis=1).plot(kind="bar", ax=ax, width=0.8, facecolor="none", edgecolor=utils.BAR_STYLE["edgecolor"], linewidth=utils.BAR_STYLE["linewidth"], legend=False)
    ax.set_title(f"State composition per dataset - {method}", **utils.TITLE_STYLE)
    ax.legend(bbox_to_anchor=(1.01, 1), loc="upper left", fontsize=utils.LEGEND_FONTSIZE, title_fontsize=utils.LEGEND_TITLE_FONTSIZE, title="State")
    ax.set_xlabel("Dataset", fontsize=utils.AXIS_FONTSIZE)
    ax.set_ylabel("Fraction of Genome", fontsize=utils.AXIS_FONTSIZE)
    ax.tick_params(axis="x", labelsize=utils.TICK_FONTSIZE, rotation=45)
    ax.tick_params(axis="y", labelsize=utils.TICK_FONTSIZE)
    fig = ax.get_figure()
    utils.widen_bars(ax)
    utils.save_fig(fig, f"out/summary_plots/composition_{method.lower().replace(' ', '_')}.pdf")

Computing state composition and lengths...


  0%|          | 0/80 [00:00<?, ?it/s]

Extracting state colors:   0%|          | 0/80 [00:00<?, ?it/s]

Saved state composition and lengths to out/df_comp.pkl


  saved out/summary_plots/avg_composition.pdf


  saved out/summary_plots/avg_mean_length.pdf


  saved out/summary_plots/avg_median_length.pdf
  saved out/summary_plots/avg_composition_stacked.pdf


  saved out/summary_plots/composition_default_chromhmm.pdf
  saved out/summary_plots/composition_joint_chromhmm.pdf


  saved out/summary_plots/composition_homer_kmeans.pdf
  saved out/summary_plots/composition_joint_homer_kmeans.pdf


  saved out/summary_plots/composition_macs2_kmeans.pdf
  saved out/summary_plots/composition_joint_macs2_kmeans.pdf


  saved out/summary_plots/composition_omnipeak_kmeans.pdf
  saved out/summary_plots/composition_joint_omnipeak_kmeans.pdf


## 4. Replicate consistency (Jaccard / Kappa / Cosine / AMI)

In [10]:
def compute_replicate_agreement():
    """rep1 vs rep2 of the same method, for every dataset that has both."""
    pairs = [(method_name, ds,
              get_method_path(ds, method_key, "rep1"),
              get_method_path(ds, method_key, "rep2"))
             for ds in DATASETS
             if os.path.isdir(f"{ds}/rep1") and os.path.isdir(f"{ds}/rep2")
             for method_key, method_name in METHOD_LABELS
             if get_method_path(ds, method_key, "rep1")
             and get_method_path(ds, method_key, "rep2")]
    return agreement_rows(pairs, "Replicates")


df_rep = utils.cached_pickle("out/df_rep.pkl", compute_replicate_agreement,
                             label="replicate consistency", valid=has_metrics)

if not df_rep.empty:
    for metric, metric_key, title in [
            (JACCARD_DISPLAY, JACCARD, "Replicate Jaccard Similarity"),
            (KAPPA_DISPLAY, KAPPA, "Replicate Cohen's Kappa"),
            (COSINE_DISPLAY, COSINE, "Replicate cosine composition"),
            (AMI_DISPLAY, AMI, "Replicate adjusted mutual information")]:
        plt.figure(figsize=(8, 6))
        sns.barplot(data=df_rep, x='Method', y=metric, order=METHOD_ORDER,
                    palette=method_palette, hue='Method', hue_order=METHOD_ORDER, dodge=False,
                    legend=False, **utils.BAR_STYLE)
        utils.hatch_joint(plt.gca(), METHOD_ORDER)
        utils.strip_points(plt.gca(), data=df_rep, x='Method', y=metric, order=METHOD_ORDER, size=2)
        plt.title(title, **utils.TITLE_STYLE)
        plt.ylabel(metric, fontsize=utils.AXIS_FONTSIZE)
        utils.group_xticks(plt.gca(), METHOD_ORDER, bands=False, fontsize=utils.TICK_FONTSIZE)
        plt.yticks(fontsize=utils.TICK_FONTSIZE)
        plt.grid(axis='y', alpha=0.3)
        yrange = plt.gca().get_ylim()[1] - plt.gca().get_ylim()[0]
        for i, method in enumerate(METHOD_ORDER):
            vals = df_rep[df_rep["Method"] == method][metric]
            if vals.empty: continue
            m, s = vals.mean(), vals.sem()
            if pd.isna(m): continue
            top = max(m + (s if not pd.isna(s) else 0), vals.max())
            plt.text(i, top + yrange * 0.01, f"{m:.2f}", ha="center", va="bottom", fontsize=utils.LABEL_FONTSIZE)
        utils.save_fig(plt.gcf(), f"out/summary_plots/replicate_{metric_key}.pdf")

Computing replicate consistency...


Replicates:   0%|          | 0/40 [00:00<?, ?it/s]

Saved replicate consistency to out/df_rep.pkl
  saved out/summary_plots/replicate_jaccard.pdf


  saved out/summary_plots/replicate_kappa.pdf
  saved out/summary_plots/replicate_cosine.pdf


  saved out/summary_plots/replicate_ami.pdf


## 4. Cross-sample concordance (Jaccard / Kappa / Cosine / AMI)

In [11]:
# The same method on two different cell lines: how much of a segmentation is
# the biology of the sample rather than the method's own signature. rep1 only,
# so the replicate variation of the section above does not enter twice, and
# with the state spaces rematched, since the two datasets were matched to
# different joint models.
def compute_cross_sample_agreement():
    """Every pair of cell lines, per method."""
    datasets = list(DATASETS)
    pairs = [(method_name, f"{ds_a}/{ds_b}",
              get_method_path(ds_a, method_key, "rep1"),
              get_method_path(ds_b, method_key, "rep1"))
             for method_key, method_name in METHOD_LABELS
             for ds_a, ds_b in combinations(datasets, 2)
             if get_method_path(ds_a, method_key, "rep1")
             and get_method_path(ds_b, method_key, "rep1")]
    return agreement_rows(pairs, "Cross-sample", rematch=True)


df_cross = utils.cached_pickle("out/df_cross_sample.pkl",
                               compute_cross_sample_agreement,
                               label="cross-sample concordance", valid=has_metrics)

if not df_cross.empty:

    for metric, title in [(JACCARD_DISPLAY, "Cross-sample Jaccard Similarity"),
                          (KAPPA_DISPLAY, "Cross-sample Cohen's Kappa"),
                          (COSINE_DISPLAY, "Cross-sample cosine composition"),
                          (AMI_DISPLAY, "Cross-sample adjusted mutual information")]:
        plt.figure(figsize=(8, 6))
        sns.barplot(data=df_cross, x='Method', y=metric, order=METHOD_ORDER,
                    palette=method_palette, hue='Method', hue_order=METHOD_ORDER, dodge=False,
                    legend=False, **utils.BAR_STYLE)
        utils.hatch_joint(plt.gca(), METHOD_ORDER)
        utils.strip_points(plt.gca(), data=df_cross, x='Method', y=metric, order=METHOD_ORDER, size=2)
        plt.title(title, **utils.TITLE_STYLE)
        plt.ylabel(metric, fontsize=utils.AXIS_FONTSIZE)
        utils.group_xticks(plt.gca(), METHOD_ORDER, bands=False, fontsize=utils.TICK_FONTSIZE)
        plt.yticks(fontsize=utils.TICK_FONTSIZE)
        plt.grid(axis='y', alpha=0.3)
        yrange = plt.gca().get_ylim()[1] - plt.gca().get_ylim()[0]
        for i, method in enumerate(METHOD_ORDER):
            vals = df_cross[df_cross["Method"] == method][metric]
            if vals.empty: continue
            m, s = vals.mean(), vals.sem()
            if pd.isna(m): continue
            top = max(m + (s if not pd.isna(s) else 0), vals.max())
            plt.text(i, top + yrange * 0.01, f"{m:.2f}", ha="center", va="bottom", fontsize=utils.LABEL_FONTSIZE)
        utils.save_fig(plt.gcf(), f"out/summary_plots/cross_sample_{utils.normalize_metric(metric)}.pdf")

Computing cross-sample concordance...


Cross-sample:   0%|          | 0/80 [00:00<?, ?it/s]

Saved cross-sample concordance to out/df_cross_sample.pkl
  saved out/summary_plots/cross_sample_jaccard.pdf


  saved out/summary_plots/cross_sample_kappa.pdf
  saved out/summary_plots/cross_sample_cosine.pdf


  saved out/summary_plots/cross_sample_ami.pdf


## 4. Individual vs Joint correspondence (Jaccard / Kappa / AMI)

In [12]:
# Each peak caller, segmented on its own dataset vs segmented jointly.
COMP_PAIRS = utils.group_methods([
    (utils.display_name(CHROMHMM_DEFAULT), CHROMHMM_DEFAULT, JOINT_CHROMHMM),
    (utils.display_name(KMEANS_HOMER), KMEANS_HOMER, JOINT_KMEANS_HOMER),
    (utils.display_name(KMEANS_MACS2), KMEANS_MACS2, JOINT_KMEANS_MACS2),
    (utils.display_name(KMEANS_OMNI), KMEANS_OMNI, JOINT_KMEANS_OMNI),
], key=lambda item: item[1])


def compute_joint_indiv_agreement():
    pairs = [(name, ds,
              get_method_path(ds, indiv_key, "rep1"),
              get_method_path(ds, joint_key, "rep1"))
             for ds in DATASETS
             for name, indiv_key, joint_key in COMP_PAIRS
             if get_method_path(ds, indiv_key, "rep1")
             and get_method_path(ds, joint_key, "rep1")]
    return agreement_rows(pairs, "Joint vs Individual")


df_ji = utils.cached_pickle("out/df_ji.pkl", compute_joint_indiv_agreement,
                            label="joint vs individual consistency", valid=has_metrics)

if not df_ji.empty:
    # The joint models have no bar of their own here, so the axis is the
    # individual side of COMP_PAIRS, grouped by caller as everywhere else.
    order = [name for name, _, _ in COMP_PAIRS]

    for metric, metric_key, title in [
            (JACCARD_DISPLAY, JACCARD, "Individual vs Joint Jaccard"),
            (KAPPA_DISPLAY, KAPPA, "Individual vs Joint Kappa"),
            (COSINE_DISPLAY, COSINE, "Individual vs joint cosine composition"),
            (AMI_DISPLAY, AMI, "Individual vs joint adjusted mutual information")]:
        plt.figure(figsize=(7, 6))
        sns.barplot(data=df_ji, x='Method', y=metric, order=order,
                    palette=method_palette, hue='Method', dodge=False,
                    legend=False, **utils.BAR_STYLE)
        utils.strip_points(plt.gca(), data=df_ji, x='Method', y=metric, order=order, size=2)
        plt.title(title, **utils.TITLE_STYLE)
        plt.ylabel(metric, fontsize=utils.AXIS_FONTSIZE)
        utils.group_xticks(plt.gca(), order, bands=False, fontsize=utils.TICK_FONTSIZE)
        plt.yticks(fontsize=utils.TICK_FONTSIZE)
        plt.grid(axis='y', alpha=0.3)
        yrange = plt.gca().get_ylim()[1] - plt.gca().get_ylim()[0]
        for i, method_name in enumerate(order):
            vals = df_ji[df_ji["Method"] == method_name][metric]
            if vals.empty: continue
            m, s = vals.mean(), vals.sem()
            if pd.isna(m): continue
            top = max(m + (s if not pd.isna(s) else 0), vals.max())
            plt.text(i, top + yrange * 0.01, f"{m:.2f}", ha="center", va="bottom", fontsize=utils.LABEL_FONTSIZE)
        utils.save_fig(plt.gcf(), f"out/summary_plots/joint_indiv_{metric_key}.pdf")

Computing joint vs individual consistency...


Joint vs Individual:   0%|          | 0/20 [00:00<?, ?it/s]

Saved joint vs individual consistency to out/df_ji.pkl
  saved out/summary_plots/joint_indiv_jaccard.pdf
  saved out/summary_plots/joint_indiv_kappa.pdf


  saved out/summary_plots/joint_indiv_cosine.pdf
  saved out/summary_plots/joint_indiv_ami.pdf


## 6. Chromatin state type interpretation


In [13]:
# The interpretation itself lives in scripts/interpretation.py: it turns the
# per-state evidence of a segmentation (bin_emissions/state_emissions.tsv,
# enrichment/enrichment.tsv, report.tsv, entropy/transition_matrix.tsv) into one
# of the chromatin state types of the table shipped next to it,
# scripts/interpretation.csv.
TYPES_TABLE = interpretation.load_interpretation_table()
TYPE_ORDER = list(TYPES_TABLE.index)                    # order used in the CSV
TYPE_SHORT = TYPES_TABLE["Short Name"].to_dict()
TYPE_COLORS = interpretation.TYPE_COLORS

print(f"Chromatin state types from {interpretation.DEFAULT_INTERPRETATION_CSV}:")
display(TYPES_TABLE)
print("Marks per type family:", interpretation.MARK_GROUPS)

Chromatin state types from /Users/Oleg.Shpynov/work/omni-chromhmm/scripts/interpretation.csv:


,Short Name,Description
Chromatin State Type,,
Promoter,Prom,Presence of promoter-associated marks H3K4me3 ...
PromoterFlanking,Prom fla,Presence of promoter-associated marks H3K4me3 ...
Enhancer,Enha,Presence of the enhancer-associated marks H3K2...
EnhancerLow,Enha low,"Same as Enhancer, but with lower signal values."
Bivalent,Biva,Presence of both activating (H3K27ac) and repr...
CTCF,CTCF,"Presence of the transcription factor CTCF, tho..."
Transcribed,Tran,Characterized by the transcription-associated ...
K9K36,K9K36,"Presence of the marks H3K9me3 and H3K36me3, a ..."
FacultativeHet,Facu,"Facultative (Polycomb) Heterochromatin, charac..."


Marks per type family: {'prom': {'H3K4me3': 1.0, 'H3K9ac': 0.8, 'H3K4me2': 0.4, 'H2AFZ': 0.3}, 'enh': {'H3K27ac': 1.0, 'H3K4me1': 1.0}, 'tx': {'H3K36me3': 1.0, 'H3K79me2': 0.4, 'H4K20me1': 0.3}, 'facu': {'H3K27me3': 1.0}, 'cons': {'H3K9me3': 1.0, 'H3K9me2': 0.4}, 'ctcf': {'CTCF': 1.0}}


In [14]:
def compute_interpretation():
    """One row per state of every segmentation analyzed by run_analysis()."""
    MATCHING_JOINT = {
        utils.display_name(CHROMHMM_DEFAULT): utils.display_name(JOINT_CHROMHMM),
        utils.display_name(KMEANS_HOMER): utils.display_name(JOINT_KMEANS_HOMER),
        utils.display_name(KMEANS_MACS2): utils.display_name(JOINT_KMEANS_MACS2),
        utils.display_name(KMEANS_OMNI): utils.display_name(JOINT_KMEANS_OMNI),
    }

    # 1. Precompute joint interpretations (once per dataset, as they are joint)
    joint_interpretations = {} # (ds, joint_method_name) -> df
    for ds in tqdm(DATASETS, desc="Joint Interpretation"):
        for method_key, method_name in METHOD_LABELS:
            if method_name in MATCHING_JOINT.values():
                # For joint models, we use rep1 to get the common interpretation
                outdir = method_outdir(ds, method_key, "rep1")
                df = interpretation.interpret_segmentation(outdir, TYPES_TABLE)
                if df is not None:
                    joint_interpretations[(ds, method_name)] = df

    # 2. Assign interpretations to all segmentations
    interpretations = []
    for ds in tqdm(DATASETS, desc="Interpretation"):
        reps = dataset_reps(ds)
        for rep in reps:
            for method_key, method_name in METHOD_LABELS:
                outdir = method_outdir(ds, method_key, rep)
                
                joint_name = MATCHING_JOINT.get(method_name)
                if joint_name:
                    # Individual method: transfer from its joint counterpart
                    joint_df = joint_interpretations.get((ds, joint_name))
                    if joint_df is None:
                        continue
                    
                    # Still need the individual report for its specific genome stats
                    ev = interpretation.load_segmentation_evidence(outdir)
                    if ev is None or ev.get("report") is None:
                        continue
                        
                    df = joint_df.copy()
                    # Preserve individual stats
                    rep_stats = ev["report"]
                    # Ensure alignment of states between individual and joint model
                    df["Fraction"] = rep_stats["total_bp"].reindex(df.index) / rep_stats["total_bp"].sum()
                    df["MedianLength"] = rep_stats["median_length"].reindex(df.index)
                    if ev.get("self_transition") is not None:
                        df["SelfTransition"] = ev["self_transition"].reindex(df.index)
                    else:
                        df["SelfTransition"] = np.nan
                else:
                    # Joint method or other: interpret normally (already computed or fresh)
                    if (ds, method_name) in joint_interpretations:
                        df = joint_interpretations[(ds, method_name)].copy()
                        # Update stats for the specific replicate if they differ
                        ev = interpretation.load_segmentation_evidence(outdir)
                        if ev is not None and ev.get("report") is not None:
                             rep_stats = ev["report"]
                             df["Fraction"] = rep_stats["total_bp"].reindex(df.index) / rep_stats["total_bp"].sum()
                             df["MedianLength"] = rep_stats["median_length"].reindex(df.index)
                             if ev.get("self_transition") is not None:
                                 df["SelfTransition"] = ev["self_transition"].reindex(df.index)
                    else:
                        df = interpretation.interpret_segmentation(outdir, TYPES_TABLE)
                
                if df is not None:
                    df = df.reset_index().rename(columns={"state": "State", "index": "State"})
                    df.insert(0, "Rep", rep)
                    df.insert(0, "Method", method_name)
                    df.insert(0, "Dataset", ds)
                    interpretations.append(df)
    return pd.concat(interpretations, ignore_index=True) if interpretations else pd.DataFrame()


df_interp = utils.cached_pickle("out/df_interp_matched.pkl", compute_interpretation,
                                label="state interpretation")

if not df_interp.empty:
    interp_tsv = "out/state_interpretation.tsv"
    df_interp.to_csv(interp_tsv, sep="\t", index=False, float_format="%.4f")
    print(f"{len(df_interp)} states of {len(df_interp.groupby(['Dataset', 'Rep', 'Method']))} "
          f"segmentations interpreted -> {interp_tsv}")

    # Interpreted type of every state, per dataset.
    for ds in DATASETS:
        for rep in ["rep1", "rep2"]:
            mat = interpretation.type_matrix(df_interp, ds, rep, methods=METHOD_ORDER)
            if mat.empty:
                continue
            print(f"\n### {ds} ({rep})")
            display(mat)

    # Types never assigned by any method — CTCF is expected here, none of these
    # datasets measures CTCF, and the table has no mark-based fallback for it.
    missing = [t for t in TYPE_ORDER if t not in set(df_interp["Type"])]
    print(f"\nTypes not assigned in any segmentation: {missing or 'none'}")


Computing state interpretation...


Joint Interpretation:   0%|          | 0/5 [00:00<?, ?it/s]

Interpretation:   0%|          | 0/5 [00:00<?, ?it/s]

Saved state interpretation to out/df_interp_matched.pkl
1200 states of 80 segmentations interpreted -> out/state_interpretation.tsv

### mcf7 (rep1)


,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15
Default ChromHMM,Enha low,Enha,Enha,Prom,Prom,Enha,Tran,Tran,Quie,Tran,K9K36,Facu,Quie,Facu,Cons
Joint ChromHMM,Enha low,Enha,Enha,Prom,Prom,Enha,Tran,Tran,Quie,Tran,K9K36,Facu,Quie,Facu,Cons
Homer KMeans,Quie,Enha,Enha,K9K36,Facu,Tran,Enha low,Tran,Tran,Tran,Prom,Prom,Cons,Enha low,Tran
Joint Homer KMeans,Quie,Enha,Enha,K9K36,Facu,Tran,Enha low,Tran,Tran,Tran,Prom,Prom,Cons,Enha low,Tran
MACS2 KMeans,Quie,Enha low,Prom,Enha,Enha,Enha low,Prom,Tran,Prom fla,Enha,Tran,Enha low,Cons,Tran,Prom
Joint MACS2 KMeans,Quie,Enha low,Prom,Enha,Enha,Enha low,Prom,Tran,Prom fla,Enha,Tran,Enha low,Cons,Tran,Prom
OmniPeak KMeans,Quie,Tran,Prom,Enha low,Facu,K9K36,Enha,Tran,Tran,Enha,Enha low,Tran,Tran,Cons,Cons
Joint OmniPeak KMeans,Quie,Tran,Prom,Enha low,Facu,K9K36,Enha,Tran,Tran,Enha,Enha low,Tran,Tran,Cons,Cons



### mcf7 (rep2)


... (14 outputs left out)

,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15
Default ChromHMM,Quie,Cons,Quie,Facu,Tran,Quie,Tran,Tran,Prom,Enha,Prom,Prom,Enha,Enha low,Enha
Joint ChromHMM,Quie,Cons,Quie,Facu,Tran,Quie,Tran,Tran,Prom,Enha,Prom,Prom,Enha,Enha low,Enha
Homer KMeans,Quie,Prom,Facu,Tran,Tran,Prom fla,Enha,Enha low,Tran,Tran,Cons,Prom,Tran,Facu,Tran
Joint Homer KMeans,Quie,Prom,Facu,Tran,Tran,Prom fla,Enha,Enha low,Tran,Tran,Cons,Prom,Tran,Facu,Tran
MACS2 KMeans,Quie,Prom,Enha low,Prom,Enha,Prom,Tran,Prom fla,Enha low,Prom fla,Prom,Enha low,Enha,Enha,Prom
Joint MACS2 KMeans,Quie,Prom,Enha low,Prom,Enha,Prom,Tran,Prom fla,Enha low,Prom fla,Prom,Enha low,Enha,Enha,Prom
OmniPeak KMeans,Quie,Facu,Tran,Enha low,Tran,Enha low,Enha,Tran,Tran,Prom,Cons,Tran,Tran,Enha,Prom
Joint OmniPeak KMeans,Quie,Facu,Tran,Enha low,Tran,Enha low,Enha,Tran,Tran,Prom,Cons,Tran,Tran,Enha,Prom



### hela_s3 (rep2)


,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15
Default ChromHMM,Quie,Cons,Quie,Facu,Tran,Quie,Tran,Tran,Prom,Enha,Prom,Prom,Enha,Enha low,Enha
Joint ChromHMM,Quie,Cons,Quie,Facu,Tran,Quie,Tran,Tran,Prom,Enha,Prom,Prom,Enha,Enha low,Enha
Homer KMeans,Quie,Prom,Facu,Tran,Tran,Prom fla,Enha,Enha low,Tran,Tran,Cons,Prom,Tran,Facu,Tran
Joint Homer KMeans,Quie,Prom,Facu,Tran,Tran,Prom fla,Enha,Enha low,Tran,Tran,Cons,Prom,Tran,Facu,Tran
MACS2 KMeans,Quie,Prom,Enha low,Prom,Enha,Prom,Tran,Prom fla,Enha low,Prom fla,Prom,Enha low,Enha,Enha,Prom
Joint MACS2 KMeans,Quie,Prom,Enha low,Prom,Enha,Prom,Tran,Prom fla,Enha low,Prom fla,Prom,Enha low,Enha,Enha,Prom
OmniPeak KMeans,Quie,Facu,Tran,Enha low,Tran,Enha low,Enha,Tran,Tran,Prom,Cons,Tran,Tran,Enha,Prom
Joint OmniPeak KMeans,Quie,Facu,Tran,Enha low,Tran,Enha low,Enha,Tran,Tran,Prom,Cons,Tran,Tran,Enha,Prom



Types not assigned in any segmentation: ['CTCF']


In [15]:
if not df_interp.empty:
    seg_keys = ["Dataset", "Rep", "Method"]

    # Plot 6a. Interpreted type of every state, per dataset
    for ds in DATASETS:
        mat = interpretation.type_matrix(df_interp, ds, "rep1", column="Type",
                                         methods=METHOD_ORDER)
        if mat.empty:
            continue
        fig, ax = plt.subplots(figsize=(0.62 * mat.shape[1] + 3, 0.42 * len(mat) + 1.6))
        for i, method in enumerate(mat.index):
            for j, state in enumerate(mat.columns):
                t = mat.iat[i, j]
                if not isinstance(t, str):
                    continue
                ax.add_patch(plt.Rectangle((j, i), 1, 1, facecolor=TYPE_COLORS[t],
                                           edgecolor="white", linewidth=1))
                ax.text(j + 0.5, i + 0.5, TYPE_SHORT[t].replace(" ", "\n"), ha="center",
                        va="center", fontsize=utils.LABEL_FONTSIZE, color=interpretation.type_text_color(t))
        ax.set_xlim(0, mat.shape[1])
        ax.set_ylim(len(mat), 0)
        ax.set_xticks(np.arange(mat.shape[1]) + 0.5)
        ax.set_xticklabels(mat.columns, fontsize=utils.TICK_FONTSIZE)
        ax.set_yticks(np.arange(len(mat)) + 0.5)
        ax.set_yticklabels(mat.index, fontsize=utils.TICK_FONTSIZE)
        ax.set_title(f"Interpreted chromatin state types - {ds} (rep1)", **utils.TITLE_STYLE)
        ax.set_xlabel("State (in the order of the segmentation)", fontsize=utils.AXIS_FONTSIZE)
        ax.tick_params(length=0)
        for spine in ax.spines.values():
            spine.set_visible(False)
        utils.save_fig(fig, f"out/summary_plots/interpretation_{ds}.pdf")

    # Genome fraction per interpreted type: types absent from a segmentation
    # contribute 0, so the bars compare methods on the same footing.
    frac = (df_interp.groupby(seg_keys + ["Type"])["Fraction"].sum()
            .unstack("Type").reindex(columns=TYPE_ORDER).fillna(0.0))
    frac_long = frac.stack().reset_index().rename(columns={0: "Fraction", "level_3": "Type"})

    # Plot 6b. Genome composition by interpreted type (stacked)
    pivot_types = frac.groupby("Method").mean().reindex(METHOD_ORDER).dropna(how="all")
    fig, ax = plt.subplots(figsize=(8, 5))
    pivot_types.plot(kind="bar", stacked=True, ax=ax, width=0.6, linewidth=0,
                     color=[TYPE_COLORS[t] for t in pivot_types.columns])
    ax.set_title("Genome composition by interpreted state type", **utils.TITLE_STYLE)
    ax.set_ylabel("Average fraction of genome", fontsize=utils.AXIS_FONTSIZE)
    ax.set_xlabel("Method", fontsize=utils.AXIS_FONTSIZE)
    ax.legend(bbox_to_anchor=(1.01, 1), loc="upper left", fontsize=utils.LEGEND_FONTSIZE, title_fontsize=utils.LEGEND_TITLE_FONTSIZE, title="Type")
    utils.group_xticks(ax, list(pivot_types.index), bands=False, fontsize=utils.TICK_FONTSIZE)
    ax.tick_params(axis="y", labelsize=utils.TICK_FONTSIZE)
    ax.grid(axis="y", alpha=0.3, linewidth=0.5)
    utils.widen_bars(ax)
    utils.save_fig(fig, "out/summary_plots/interpretation_composition.pdf")

    # Plot 6c. Genome fraction per interpreted type, without Quiescent
    signal = frac_long[frac_long["Type"] != "Quiescent"]
    types_present = [t for t in TYPE_ORDER
                     if t != "Quiescent" and signal.loc[signal["Type"] == t, "Fraction"].sum() > 0]
    plt.figure(figsize=(max(12, 1.7 * len(types_present)), 5))
    ax = plt.gca()
    sns.barplot(data=signal, x="Type", y="Fraction", hue="Method", order=types_present,
                hue_order=METHOD_ORDER, palette=method_palette, ax=ax, **utils.BAR_STYLE)
    utils.hatch_joint(ax, METHOD_ORDER)
    utils.strip_points(ax, data=signal, x="Type", y="Fraction", hue="Method",
                       order=types_present, hue_order=METHOD_ORDER, size=2, alpha=0.5, jitter=0.2)
    ax.set_title("Fraction of genome per interpreted state type", **utils.TITLE_STYLE)
    ax.set_ylabel("Fraction of genome", fontsize=utils.AXIS_FONTSIZE)
    ax.set_xlabel("Interpreted state type", fontsize=utils.AXIS_FONTSIZE)
    ax.grid(axis="y", alpha=0.3, linewidth=0.5)
    ax.tick_params(axis="x", labelsize=utils.TICK_FONTSIZE, rotation=30)
    ax.tick_params(axis="y", labelsize=utils.TICK_FONTSIZE)
    ax.legend(title="Method", fontsize=utils.LEGEND_FONTSIZE, title_fontsize=utils.LEGEND_TITLE_FONTSIZE, bbox_to_anchor=(1.01, 1),
              loc="upper left", borderaxespad=0)
    utils.save_fig(plt.gcf(), "out/summary_plots/interpretation_fraction.pdf")

    # Plot 6d. How many states each method spends on every type
    counts = (df_interp.groupby(seg_keys + ["Type"]).size().unstack("Type")
              .reindex(columns=TYPE_ORDER).fillna(0).stack().reset_index()
              .rename(columns={0: "States", "level_3": "Type"}))
    plt.figure(figsize=(max(12, 1.7 * len(TYPE_ORDER)), 5))
    ax = plt.gca()
    sns.barplot(data=counts, x="Type", y="States", hue="Method", order=TYPE_ORDER,
                hue_order=METHOD_ORDER, palette=method_palette, ax=ax, **utils.BAR_STYLE)
    utils.hatch_joint(ax, METHOD_ORDER)
    utils.strip_points(ax, data=counts, x="Type", y="States", hue="Method", order=TYPE_ORDER,
                       hue_order=METHOD_ORDER, size=2, alpha=0.5, jitter=0.2)
    ax.set_title("Number of states per interpreted type", **utils.TITLE_STYLE)
    ax.set_ylabel("States per segmentation", fontsize=utils.AXIS_FONTSIZE)
    ax.set_xlabel("Interpreted state type", fontsize=utils.AXIS_FONTSIZE)
    ax.grid(axis="y", alpha=0.3, linewidth=0.5)
    ax.tick_params(axis="x", labelsize=utils.TICK_FONTSIZE, rotation=30)
    ax.tick_params(axis="y", labelsize=utils.TICK_FONTSIZE)
    ax.legend(title="Method", fontsize=utils.LEGEND_FONTSIZE, title_fontsize=utils.LEGEND_TITLE_FONTSIZE, bbox_to_anchor=(1.01, 1),
              loc="upper left", borderaxespad=0)
    utils.save_fig(plt.gcf(), "out/summary_plots/interpretation_states_per_type.pdf")

    # Plot 6e. Interpretability per method: types covered, and how clean the calls are
    summary = (df_interp.groupby(seg_keys)
               .agg(Types=("Type", "nunique"), Margin=("Margin", "mean")).reset_index())
    fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
    for ax, col, title, ylabel in [
        (axes[0], "Types", "Distinct state types recovered", "Types per segmentation"),
        (axes[1], "Margin", "Interpretation margin (higher = less mixed marks)", "Mean margin"),
    ]:
        sns.barplot(data=summary, x="Method", y=col, order=METHOD_ORDER, hue="Method",
                    hue_order=METHOD_ORDER, palette=method_palette, dodge=False, ax=ax,
                    legend=False, **utils.BAR_STYLE)
        utils.hatch_joint(ax, METHOD_ORDER)
        utils.strip_points(ax, data=summary, x="Method", y=col, order=METHOD_ORDER,
                           dodge=False, size=3, alpha=0.6)
        ax.set_title(title, **utils.TITLE_STYLE)
        ax.set_ylabel(ylabel, fontsize=utils.AXIS_FONTSIZE)
        ax.set_xlabel("")
        ax.tick_params(axis="y", labelsize=utils.TICK_FONTSIZE)
        utils.group_xticks(ax, METHOD_ORDER, bands=False, fontsize=utils.TICK_FONTSIZE)
        ax.grid(axis="y", alpha=0.3, linewidth=0.5)
    utils.save_fig(fig, "out/summary_plots/interpretation_summary.pdf")

    print("Average fraction of genome per interpreted type:")
    display(pivot_types.round(4))


  saved out/summary_plots/interpretation_mcf7.pdf


  saved out/summary_plots/interpretation_gm12878.pdf


  saved out/summary_plots/interpretation_k562.pdf


  saved out/summary_plots/interpretation_cd14_monocyte.pdf


  saved out/summary_plots/interpretation_hela_s3.pdf


  saved out/summary_plots/interpretation_composition.pdf


  saved out/summary_plots/interpretation_fraction.pdf


  saved out/summary_plots/interpretation_states_per_type.pdf
  saved out/summary_plots/interpretation_summary.pdf
Average fraction of genome per interpreted type:


Type,Promoter,PromoterFlanking,Enhancer,EnhancerLow,Bivalent,CTCF,Transcribed,K9K36,FacultativeHet,ConstitutiveHet,Quiescent
Method,,,,,,,,,,,
Default ChromHMM,0.0096,0.0248,0.0162,0.0141,0.0008,0.0,0.0639,0.0006,0.0440,0.0326,0.7936
Joint ChromHMM,0.0113,0.0047,0.0170,0.0140,0.0010,0.0,0.0671,0.0005,0.0291,0.0151,0.8401
Homer KMeans,0.0156,0.0075,0.0237,0.0190,0.0000,0.0,0.0767,0.0010,0.0307,0.0186,0.8071
Joint Homer KMeans,0.0159,0.0084,0.0234,0.0183,0.0000,0.0,0.0754,0.0010,0.0313,0.0179,0.8083
MACS2 KMeans,0.0083,0.0039,0.0085,0.0129,0.0000,0.0,0.0035,0.0000,0.0008,0.0013,0.9607
Joint MACS2 KMeans,0.0086,0.0042,0.0087,0.0127,0.0000,0.0,0.0031,0.0000,0.0008,0.0013,0.9607
OmniPeak KMeans,0.0116,0.0053,0.0151,0.0229,0.0000,0.0,0.0866,0.0011,0.0417,0.0212,0.7945
Joint OmniPeak KMeans,0.0117,0.0055,0.0125,0.0241,0.0000,0.0,0.0868,0.0012,0.0418,0.0217,0.7946


In [16]:
# 6f. Genome composition per method, over the interpreted state types.
#
# The raw state names cannot carry this plot: process_sagaconf.sh matched every
# model to the joint model of its own dataset, and each of those numbers its
# states arbitrarily, so state "5" of MCF7 and state "5" of K562 are different
# states and the mean of a state over the datasets means nothing - which is
# what avg_composition.pdf above shows. The interpreted types are the same
# vocabulary in every dataset, so the composition is read over them instead.
# One bar per method, the joint models hatched; there is no reference bar,
# ref_chromhmm is null for every SAGAconf dataset. Plot 6b above is the
# stacked form of the same numbers.
if not df_interp.empty:
    # Types absent from a segmentation contribute 0, so the bars compare the
    # methods on the same footing, and a type no method ever calls is left out.
    frac_by_type = (df_interp.groupby(["Dataset", "Rep", "Method", "Type"])["Fraction"]
                    .sum().unstack("Type").reindex(columns=TYPE_ORDER).fillna(0.0))
    types_present = [t for t in TYPE_ORDER if frac_by_type[t].sum() > 0]

    # Broken between BREAK_LOW and BREAK_HIGH, as every composition plot of the
    # project is: Quiescent covers most of the genome and flattens the rest of
    # the types on a shared axis.
    utils.broken_bar_plot(
        frac_by_type[types_present].stack().reset_index(name="Fraction"),
        "Type", "Fraction", order=types_present, hue="Method",
        hue_order=METHOD_ORDER, palette=method_palette,
        break_low=BREAK_LOW, break_high=BREAK_HIGH,
        figsize=(max(12, len(types_present) * len(METHOD_ORDER) * 0.3), 6),
        title="Genome composition by interpreted state type",
        xlabel="Interpreted state type", ylabel="Fraction of genome",
        legend=True, points={"size": 2, "alpha": 0.5, "jitter": 0.2},
        path="out/summary_plots/composition_methods.pdf")


  saved out/summary_plots/composition_methods.pdf


In [17]:
# 6f. Replicate consistency on interpreted state types


def interpretation_map():
    """(Dataset, Method, Rep, State) -> interpreted Type."""
    return df_interp.set_index(["Dataset", "Method", "Rep", "State"])["Type"].to_dict()


def type_agreement_pair(side1, side2, interp_map):
    """Two segmentations reduced to interpreted state types.

    Each side is a (dataset, method_key, method_name, rep) tuple. Returns the
    (overlap, lengths1, lengths2) triple that match.agreement_metrics() and
    match.per_state_agreement() take, keyed by interpreted type instead of by
    state, or None when either side is not on disk.

    Reducing to types is also what lets two *different* datasets be compared
    without emission rematching: process_sagaconf.sh matches every model to the
    joint model of its own dataset, so state "13" of MCF7 and state "13" of
    K562 are unrelated names, while the interpreted types are a vocabulary
    every segmentation shares.
    """
    (ds1, key1, name1, rep1), (ds2, key2, name2, rep2) = side1, side2
    p1 = get_method_path(ds1, key1, rep1)
    p2 = get_method_path(ds2, key2, rep2)
    if not p1 or not p2 or not os.path.exists(p1) or not os.path.exists(p2):
        return None

    s1, s2 = match.load_bed(p1), match.load_bed(p2)
    l1, l2 = match.state_lengths(s1), match.state_lengths(s2)
    # pair_overlap() puts its *work* state first, so side2 goes in as the
    # reference and the keys come out (side1_state, side2_state).
    overlap = match.pair_overlap(s2, s1)

    # Interpreted type of every state, per side. A state the interpretation
    # never reached counts as background rather than as a type of its own.
    m1 = {s: interp_map.get((ds1, name1, rep1, str(s)), QUIESCENT) for s in l1}
    m2 = {s: interp_map.get((ds2, name2, rep2, str(s)), QUIESCENT) for s in l2}

    type_overlap = defaultdict(float)
    for (st1, st2), val in overlap.items():
        type_overlap[(m1[st1], m2[st2])] += val

    type_l1, type_l2 = defaultdict(float), defaultdict(float)
    for st1, val in l1.items():
        type_l1[m1[st1]] += val
    for st2, val in l2.items():
        type_l2[m2[st2]] += val

    return type_overlap, type_l1, type_l2


def type_agreement_inputs(ds, method_key, method_name, interp_map):
    """rep1 / rep2 of one segmentation, reduced to interpreted state types."""
    return type_agreement_pair((ds, method_key, method_name, "rep1"),
                               (ds, method_key, method_name, "rep2"),
                               interp_map)


def type_agreement_rows(pairs, desc, interp_map):
    """DataFrame of the agreement over interpreted types of every
    (method_name, label, side1, side2) pair."""
    rows = []
    for method_name, label, side1, side2 in tqdm(pairs, desc=desc):
        inputs = type_agreement_pair(side1, side2, interp_map)
        if inputs is None:
            continue
        metrics = match.agreement_metrics(*inputs)
        if metrics:
            rows.append({"Method": method_name, "Dataset": label,
                         **{utils.metric_display(m): v for m, v in metrics.items()}})
    df = pd.DataFrame(rows)
    df.attrs["version"] = AGREEMENT_CACHE_VERSION
    return df


def datasets_with_replicates():
    """Datasets carrying both replicate folders on disk."""
    return [ds for ds in DATASETS
            if os.path.isdir(f"{ds}/rep1") and os.path.isdir(f"{ds}/rep2")]


def compute_replicate_agreement_types():
    """rep1 vs rep2 of the same method, over the interpreted state types."""
    if df_interp.empty:
        return pd.DataFrame(columns=["Method", "Dataset", JACCARD_DISPLAY, KAPPA_DISPLAY,
                                     COSINE_DISPLAY, AMI_DISPLAY])
    pairs = [(method_name, ds,
              (ds, method_key, method_name, "rep1"),
              (ds, method_key, method_name, "rep2"))
             for ds in datasets_with_replicates()
             for method_key, method_name in METHOD_LABELS]
    return type_agreement_rows(pairs, "Replicates per type", interpretation_map())


def compute_replicate_agreement_per_type():
    """rep1 vs rep2 agreement for each interpreted state type.

    One row per type a method actually emits: match.per_state_agreement() leaves
    out the types absent from both replicates instead of scoring them a perfect
    1.0, which is what an empty intersection over an empty union used to give.

    There is no cosine per type: match.per_state_agreement() reports Jaccard
    and kappa only, and a composition vector of a single type has one
    component, so its cosine would be 1.0 by construction.
    """
    if df_interp.empty:
        return pd.DataFrame(columns=["Method", "Dataset", "Type", JACCARD_DISPLAY, KAPPA_DISPLAY])

    interp_map = interpretation_map()

    rows = []
    for ds in tqdm(datasets_with_replicates(), desc="Replicates per type"):
        for method_key, method_name in METHOD_LABELS:
            inputs = type_agreement_inputs(ds, method_key, method_name, interp_map)
            if inputs is None:
                continue
            for t, metrics in match.per_state_agreement(*inputs).items():
                rows.append({"Method": method_name, "Dataset": ds, "Type": t,
                             **{utils.metric_display(m): v for m, v in metrics.items()}})

    df = pd.DataFrame(rows)
    return df


df_rep_types = utils.cached_pickle("out/df_rep_types.pkl", compute_replicate_agreement_types,
                                   label="replicate consistency on types", valid=has_metrics)
df_rep_per_type = utils.cached_pickle("out/df_rep_per_type.pkl", compute_replicate_agreement_per_type,
                                      label="replicate consistency per type", valid=nonempty)

if not df_rep_per_type.empty:
    # Only the types some method emits; an absent type no longer scores 1.0, so
    # forcing the background trio in here would just add empty x positions.
    types_order = [t for t in TYPE_ORDER if t in set(df_rep_per_type["Type"])]
    for metric, metric_key in [(JACCARD_DISPLAY, JACCARD),
                               (KAPPA_DISPLAY, KAPPA)]:
        plt.figure(figsize=(max(12, len(types_order) * 2), 6))
        ax = sns.barplot(data=df_rep_per_type, x="Type", y=metric, hue="Method",
                         order=types_order, hue_order=METHOD_ORDER, palette=method_palette,
                         **utils.BAR_STYLE)
        utils.hatch_joint(ax, METHOD_ORDER)
        utils.strip_points(ax, data=df_rep_per_type, x="Type", y=metric, hue="Method",
                           order=types_order, hue_order=METHOD_ORDER, size=2.5, alpha=0.6, jitter=0.2)
        
        ax.set_title(f"Replicate {metric} per interpreted state type", **utils.TITLE_STYLE)
        ax.set_ylabel(metric, fontsize=utils.AXIS_FONTSIZE)
        ax.set_xlabel("Interpreted Type", fontsize=utils.AXIS_FONTSIZE)
        ax.set_ylim(0 if metric_key == JACCARD else None, 1.05)
        ax.grid(axis='y', alpha=0.3)
        plt.xticks(rotation=45, ha="right", fontsize=utils.TICK_FONTSIZE)
        plt.yticks(fontsize=utils.TICK_FONTSIZE)
        ax.legend(title="Method", fontsize=utils.LEGEND_FONTSIZE, title_fontsize=utils.LEGEND_TITLE_FONTSIZE, bbox_to_anchor=(1.01, 1), loc="upper left")

        utils.save_fig(plt.gcf(), f"out/summary_plots/interpretation_replicate_{metric_key}_per_type.pdf")

        # Combined across methods
        plt.figure(figsize=(max(12, len(types_order) * 2), 6))
        ax = sns.barplot(data=df_rep_per_type, x="Type", y=metric,
                         order=types_order, color="#4878CF",
                         **utils.BAR_STYLE)
        utils.strip_points(ax, data=df_rep_per_type, x="Type", y=metric,
                           order=types_order, size=2.5, alpha=0.6, jitter=0.2)
        ax.set_title(f"Average Replicate {metric} per interpreted type (combined across methods)",
                     **utils.TITLE_STYLE)
        ax.set_ylabel(metric, fontsize=utils.AXIS_FONTSIZE)
        ax.set_xlabel("Interpreted Type", fontsize=utils.AXIS_FONTSIZE)
        ax.set_ylim(0 if metric_key == JACCARD else None, 1.05)
        ax.grid(axis='y', alpha=0.3)
        plt.xticks(rotation=45, ha="right", fontsize=utils.TICK_FONTSIZE)
        plt.yticks(fontsize=utils.TICK_FONTSIZE)
        utils.save_fig(plt.gcf(), f"out/summary_plots/interpretation_replicate_{metric_key}_per_type_combined.pdf")

if not df_rep_types.empty:
    for metric, metric_key in [(JACCARD_DISPLAY, JACCARD),
                               (KAPPA_DISPLAY, KAPPA),
                               (COSINE_DISPLAY, COSINE),
                               (AMI_DISPLAY, AMI)]:
        plt.figure(figsize=(8, 6))
        sns.barplot(data=df_rep_types, x='Method', y=metric, order=METHOD_ORDER,
                    palette=method_palette, hue='Method', hue_order=METHOD_ORDER, dodge=False,
                    legend=False, **utils.BAR_STYLE)
        utils.hatch_joint(plt.gca(), METHOD_ORDER)
        utils.strip_points(plt.gca(), data=df_rep_types, x='Method', y=metric, order=METHOD_ORDER, size=2)
        plt.title(f"Replicate {metric} (interpreted state types)", **utils.TITLE_STYLE)
        plt.ylabel(metric, fontsize=utils.AXIS_FONTSIZE)
        utils.group_xticks(plt.gca(), METHOD_ORDER, bands=False, fontsize=utils.TICK_FONTSIZE)
        plt.yticks(fontsize=utils.TICK_FONTSIZE)
        plt.grid(axis='y', alpha=0.3)

        yrange = plt.gca().get_ylim()[1] - plt.gca().get_ylim()[0]
        for i, method in enumerate(METHOD_ORDER):
            vals = df_rep_types[df_rep_types["Method"] == method][metric]
            if vals.empty: continue
            m, s = vals.mean(), vals.sem()
            if pd.isna(m): continue
            top = max(m + (s if not pd.isna(s) else 0), vals.max())
            plt.text(i, top + yrange * 0.01, f"{m:.2f}", ha="center", va="bottom", fontsize=utils.LABEL_FONTSIZE)

        utils.save_fig(plt.gcf(), f"out/summary_plots/interpretation_replicate_{metric_key}.pdf")

Computing replicate consistency on types...


Replicates per type:   0%|          | 0/40 [00:00<?, ?it/s]

Saved replicate consistency on types to out/df_rep_types.pkl
Computing replicate consistency per type...


Replicates per type:   0%|          | 0/5 [00:00<?, ?it/s]

Saved replicate consistency per type to out/df_rep_per_type.pkl


  saved out/summary_plots/interpretation_replicate_jaccard_per_type.pdf
  saved out/summary_plots/interpretation_replicate_jaccard_per_type_combined.pdf


  saved out/summary_plots/interpretation_replicate_kappa_per_type.pdf
  saved out/summary_plots/interpretation_replicate_kappa_per_type_combined.pdf


  saved out/summary_plots/interpretation_replicate_jaccard.pdf
  saved out/summary_plots/interpretation_replicate_kappa.pdf


  saved out/summary_plots/interpretation_replicate_cosine.pdf
  saved out/summary_plots/interpretation_replicate_ami.pdf


## 6g. Cross-sample and individual vs joint concordance on interpreted state types


In [18]:
# Sections 4 and 5 above, read over the interpreted state types: the same two
# comparisons, with every state reduced to the type it was interpreted as.
#
# The composition (cosine) leaves the biggest state of either side out
# (match.composition_cosine()) - Quiescent, most of the genome, would otherwise
# dominate both vectors and push every cosine to 1.
def compute_cross_sample_agreement_types():
    """Every pair of cell lines, per method, over the interpreted types. rep1
    only, so the replicate variation of 6f does not enter twice."""
    if df_interp.empty:
        return pd.DataFrame(columns=["Method", "Dataset", JACCARD_DISPLAY,
                                     KAPPA_DISPLAY, COSINE_DISPLAY, AMI_DISPLAY])
    datasets = list(DATASETS)
    pairs = [(method_name, f"{ds_a}/{ds_b}",
              (ds_a, method_key, method_name, "rep1"),
              (ds_b, method_key, method_name, "rep1"))
             for method_key, method_name in METHOD_LABELS
             for ds_a, ds_b in combinations(datasets, 2)]
    return type_agreement_rows(pairs, "Cross-sample per type", interpretation_map())


def compute_joint_indiv_agreement_types():
    """Each peak caller on its own dataset vs segmented jointly, over the
    interpreted types."""
    if df_interp.empty:
        return pd.DataFrame(columns=["Method", "Dataset", JACCARD_DISPLAY,
                                     KAPPA_DISPLAY, COSINE_DISPLAY, AMI_DISPLAY])
    pairs = [(name, ds,
              (ds, indiv_key, name, "rep1"),
              (ds, joint_key, utils.display_name(joint_key), "rep1"))
             for ds in DATASETS
             for name, indiv_key, joint_key in COMP_PAIRS]
    return type_agreement_rows(pairs, "Joint vs Individual per type",
                               interpretation_map())


df_cross_types = utils.cached_pickle("out/df_cross_sample_types.pkl",
                                     compute_cross_sample_agreement_types,
                                     label="cross-sample concordance on types",
                                     valid=has_metrics)
df_ji_types = utils.cached_pickle("out/df_ji_types.pkl",
                                  compute_joint_indiv_agreement_types,
                                  label="joint vs individual consistency on types",
                                  valid=has_metrics)

# The joint models have no individual-vs-joint bar of their own, so that panel
# is drawn over the four callers only, the way section 5 draws it.
JI_ORDER = [name for name, _, _ in COMP_PAIRS]

for df, order, stem, title_prefix in [
        (df_cross_types, METHOD_ORDER, "interpretation_cross_sample", "Cross-sample"),
        (df_ji_types, JI_ORDER, "interpretation_joint_indiv", "Individual vs joint")]:
    if df.empty:
        continue
    for metric, metric_key in [(JACCARD_DISPLAY, JACCARD),
                               (KAPPA_DISPLAY, KAPPA),
                               (COSINE_DISPLAY, COSINE),
                               (AMI_DISPLAY, AMI)]:
        plt.figure(figsize=(8, 6))
        sns.barplot(data=df, x='Method', y=metric, order=order,
                    palette=method_palette, hue='Method', hue_order=order, dodge=False,
                    legend=False, **utils.BAR_STYLE)
        utils.hatch_joint(plt.gca(), order)
        utils.strip_points(plt.gca(), data=df, x='Method', y=metric, order=order, size=2)
        plt.title(f"{title_prefix} {metric} (interpreted state types)", **utils.TITLE_STYLE)
        plt.ylabel(metric, fontsize=utils.AXIS_FONTSIZE)
        utils.group_xticks(plt.gca(), order, bands=False, fontsize=utils.TICK_FONTSIZE)
        plt.yticks(fontsize=utils.TICK_FONTSIZE)
        plt.grid(axis='y', alpha=0.3)
        yrange = plt.gca().get_ylim()[1] - plt.gca().get_ylim()[0]
        for i, method in enumerate(order):
            vals = df[df["Method"] == method][metric]
            if vals.empty: continue
            m, s = vals.mean(), vals.sem()
            if pd.isna(m): continue
            top = max(m + (s if not pd.isna(s) else 0), vals.max())
            plt.text(i, top + yrange * 0.01, f"{m:.2f}", ha="center", va="bottom", fontsize=utils.LABEL_FONTSIZE)
        utils.save_fig(plt.gcf(), f"out/summary_plots/{stem}_{metric_key}.pdf")

Computing cross-sample concordance on types...


Cross-sample per type:   0%|          | 0/80 [00:00<?, ?it/s]

Saved cross-sample concordance on types to out/df_cross_sample_types.pkl
Computing joint vs individual consistency on types...


Joint vs Individual per type:   0%|          | 0/20 [00:00<?, ?it/s]

Saved joint vs individual consistency on types to out/df_ji_types.pkl
  saved out/summary_plots/interpretation_cross_sample_jaccard.pdf


  saved out/summary_plots/interpretation_cross_sample_kappa.pdf
  saved out/summary_plots/interpretation_cross_sample_cosine.pdf


  saved out/summary_plots/interpretation_cross_sample_ami.pdf
  saved out/summary_plots/interpretation_joint_indiv_jaccard.pdf
  saved out/summary_plots/interpretation_joint_indiv_kappa.pdf


  saved out/summary_plots/interpretation_joint_indiv_cosine.pdf
  saved out/summary_plots/interpretation_joint_indiv_ami.pdf
